# Cycle 4: final validation predictions

This notebook **refits CatBoost and regenerates `validation-predictions.csv` when run from top to bottom**. It contains the actual training, prediction, template-filling, saving, and checking steps, rather than just loading the previous result.

We reuse the selected CatBoost configuration from Cycle 2: depth 6, MAE training loss, and 94 trees. We now fit that fixed configuration on all 48,000 labeled January-October loads and predict the 12,000 November-December loads in `validation.csv`.

The result keeps the two template columns, `load_id` and `predicted_rate`, in the template's original row order. Prices are saved to two decimal places.

**Run:** choose the **Python (spotter)** kernel and run all cells in order. Generated final artifacts are refreshed. The supplied CSVs, original template, and earlier evaluation artifacts are checked for changes and preserved. The reusable feature-preparation classes come from `freight_model.py`; the workflow itself is visible below.

In [1]:
from pathlib import Path
import sys
import hashlib
import json
from time import perf_counter

import catboost
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from IPython.display import display, Markdown

ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / 'freight_model.py').is_file() and (folder / 'train-test.csv').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the spotter project.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from freight_model import FreightPreprocessor, FreightRatePipeline, load_development, source_hashes
from score import validate_predictions

DESTINATION = ROOT / 'validation-predictions.csv'
ARTIFACT_DIR = ROOT / 'outputs' / 'final_catboost'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Python {sys.version.split()[0]} | CatBoost {catboost.__version__}')
print('Output CSV:', DESTINATION)

Python 3.13.7 | CatBoost 1.2.10
Output CSV: C:\Users\default.LAPTOP-H1OG6SJ2\Desktop\Projects\spotter\validation-predictions.csv


## 1. Record what must stay unchanged

A SHA-256 hash acts as a file fingerprint. Comparing the fingerprint before and after this run checks that the supplied data and earlier experiments were not overwritten. The new final model lives in its own directory, `outputs/final_catboost/`.

The existing prediction CSV is a generated result, so this notebook intentionally regenerates it. We record its previous fingerprint to check reproducibility after the run.

In [2]:
def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

original_hashes = source_hashes(ROOT)
preserved_files = [
    path
    for name in ['catboost', 'xgboost', 'ffn', 'comparison']
    for path in (ROOT / 'outputs' / name).rglob('*')
    if path.is_file()
]
evaluation_hashes = {str(path.relative_to(ROOT)): file_hash(path) for path in preserved_files}
previous_output_hash = file_hash(DESTINATION) if DESTINATION.exists() else None

configuration = json.loads((ROOT / 'outputs/catboost/metadata.json').read_text(encoding='utf-8'))
parameters = configuration['parameters'].copy()
assert int(parameters['iterations']) == int(configuration['selected_trees'])

display(pd.DataFrame([
    {'setting':'Model', 'value':configuration['selected_model']},
    {'setting':'Tree depth', 'value':parameters['depth']},
    {'setting':'Trees', 'value':parameters['iterations']},
    {'setting':'Loss function', 'value':parameters['loss_function']},
    {'setting':'Learning rate', 'value':parameters['learning_rate']},
    {'setting':'Random seed', 'value':parameters['random_seed']},
]))
print(f'Fingerprints recorded for four source CSVs and {len(evaluation_hashes)} earlier result files.')

,setting,value
0,Model,CatBoost depth 6 MAE
1,Tree depth,6
2,Trees,94
3,Loss function,MAE
4,Learning rate,0.05
5,Random seed,42


Fingerprints recorded for four source CSVs and 47 earlier result files.


## 2. Load the labeled data, prediction inputs, and answer template

The names can be confusing: `validation.csv` has **no known target prices**. It is the final input file, not the development set used for model selection. The development and local-test measurements are documented in Cycles 2 and 3.

Before training, we check that IDs are unique, the input and template IDs match, and the prediction dates come after the labeled dates. Matching by ID avoids assigning a price to the wrong load if the input rows and template rows are ordered differently.

In [3]:
labeled = load_development(ROOT)  # sorts by date and load_id
inputs = pd.read_csv(ROOT / 'validation.csv', parse_dates=['date'])
template = pd.read_csv(ROOT / 'validation-predictions-template.csv')

if list(template.columns) != ['load_id', 'predicted_rate']:
    raise ValueError('Unexpected template column names or order.')
for name, frame in [('validation inputs', inputs), ('template', template)]:
    if frame['load_id'].isna().any() or not frame['load_id'].is_unique:
        raise ValueError(f'Missing or duplicated load_id in {name}.')
if set(inputs['load_id']) != set(template['load_id']):
    raise ValueError('The validation inputs and template contain different IDs.')
if inputs['date'].isna().any() or labeled['date'].max() >= inputs['date'].min():
    raise ValueError('Expected prediction dates to follow all labeled training dates.')

display(pd.DataFrame([
    {'file':'train-test.csv', 'rows':len(labeled),
     'first_date':labeled['date'].min().date(), 'last_date':labeled['date'].max().date(),
     'purpose':'Fit the final model; posted_rate is available'},
    {'file':'validation.csv', 'rows':len(inputs),
     'first_date':inputs['date'].min().date(), 'last_date':inputs['date'].max().date(),
     'purpose':'Predict; true posted_rate is unavailable'},
    {'file':'validation-predictions-template.csv', 'rows':len(template),
     'first_date':None, 'last_date':None, 'purpose':'Defines output columns and row order'},
]))
display(template.head())

,file,rows,first_date,last_date,purpose
0,train-test.csv,48000,2025-01-01,2025-10-31,Fit the final model; posted_rate is available
1,validation.csv,12000,2025-11-01,2025-12-31,Predict; true posted_rate is unavailable
2,validation-predictions-template.csv,12000,None,None,Defines output columns and row order


,load_id,predicted_rate
0,TE-000001,NaN
1,TE-000002,NaN
2,TE-000003,NaN
3,TE-000004,NaN
4,TE-000005,NaN


## 3. Refit the fixed CatBoost pipeline on all labeled rows

The fitted pipeline corrects negative weight signs in memory, preserves missing numerical values for CatBoost's native handling, retains missing/sign flags, keeps coordinates separate, prepares date features, and handles city/equipment categories. It learns any city-coordinate lookup only from labeled fitting rows. `load_id`, `posted_rate`, output placeholders, `quote_signal`, and `market_index` are excluded from features.

There is **no new tuning and no early-stopping set here**. The 94-tree configuration was selected earlier. Using all labeled rows is appropriate for this final prediction step because the future target prices are not available to the model.

The earlier September-October scores belong to the earlier January-August evaluation model. They are not a measured score for this refitted model, which now also learns from September-October.

In [4]:
pipeline = FreightRatePipeline([
    ('features', FreightPreprocessor()),
    ('model', CatBoostRegressor(**parameters)),
])
started = perf_counter()
pipeline.fit(labeled, labeled['posted_rate'])
training_seconds = perf_counter() - started
assert pipeline.named_steps['model'].tree_count_ == int(parameters['iterations'])

print(f'Refit completed on {len(labeled):,} labeled loads in {training_seconds:.1f} seconds.')
print('Fitted through:', labeled['date'].max().date())
print('Trees:', pipeline.named_steps['model'].tree_count_)
display(pd.DataFrame({'model_feature': pipeline.named_steps['features'].get_feature_names_out()}))

Refit completed on 48,000 labeled loads in 6.5 seconds.
Fitted through: 2025-10-31
Trees: 94


,model_feature
0,pickup
1,delivery
2,equipment
3,distance
4,weight
5,weight_is_missing
6,weight_was_negative
7,pickup_lat
8,pickup_lon
9,delivery_lat


In [5]:
# Show the in-memory weight correction and confirm that missingness is retained.
weight_audit = []
for name, frame in [('Labeled fitting rows', labeled), ('Validation inputs', inputs)]:
    prepared = pipeline.named_steps['features'].transform(frame)
    assert not prepared['weight'].dropna().lt(0).any()
    assert prepared['weight'].isna().equals(frame['weight'].isna())
    weight_audit.append({
        'dataset': name,
        'negative_weights_in_source': int(frame['weight'].lt(0).sum()),
        'negative_weights_after_preparation': int(prepared['weight'].lt(0).sum()),
        'missing_weights_retained': int(prepared['weight'].isna().sum()),
    })
display(pd.DataFrame(weight_audit))

,dataset,negative_weights_in_source,negative_weights_after_preparation,missing_weights_retained
0,Labeled fitting rows,292,0,300
1,Validation inputs,145,0,165


## 4. Predict and fill a copy of the template by load ID

`pipeline.predict` accepts the raw input columns and applies the same fitted preprocessing automatically. A fixed $0.01 floor enforces positive output prices, as in the evaluation pipelines.

The template's existing order is authoritative. We build an ID-to-prediction mapping, then use it to fill `predicted_rate` on a copy of the template. The original template remains unchanged.

In [6]:
predictions = pipeline.predict(inputs)
assert len(predictions) == len(inputs)
assert np.isfinite(predictions).all() and (predictions > 0).all()

prediction_by_id = pd.Series(predictions, index=inputs['load_id'])
filled = template.copy()
filled['predicted_rate'] = filled['load_id'].map(prediction_by_id)
validate_predictions(filled)
pd.testing.assert_series_equal(filled['load_id'], template['load_id'])

print(f'Filled {len(filled):,} rows with no missing predictions.')
display(filled.head(10).round({'predicted_rate':2}))
display(filled.tail(5).round({'predicted_rate':2}))

Filled 12,000 rows with no missing predictions.


,load_id,predicted_rate
0,TE-000001,871.63
1,TE-000002,4906.43
2,TE-000003,5053.93
3,TE-000004,4109.71
4,TE-000005,1861.78
5,TE-000006,4286.89
6,TE-000007,5182.98
7,TE-000008,2826.29
8,TE-000009,1087.91
9,TE-000010,1133.72


,load_id,predicted_rate
11995,TE-011996,884.66
11996,TE-011997,1543.57
11997,TE-011998,952.39
11998,TE-011999,3405.04
11999,TE-012000,567.34


## 5. Save and reload the final model; export the requested CSV

The complete model is saved to `outputs/final_catboost/pipeline.joblib`, including its preprocessing. Reloading it must reproduce the predictions.

The CSV is written to **`validation-predictions.csv`** in the project root. It contains exactly two columns and saves dollar prices to two decimal places. We reload the CSV and use the provided `score.py` validator to check the required IDs, row count, schema, and finite positive prices. This validator checks file correctness; it cannot measure accuracy without the hidden true prices.

In [7]:
model_path = ARTIFACT_DIR / 'pipeline.joblib'
joblib.dump(pipeline, model_path)
restored = joblib.load(model_path)
np.testing.assert_allclose(restored.predict(inputs), predictions, rtol=0, atol=1e-10)

filled.to_csv(DESTINATION, index=False, float_format='%.2f')
saved = pd.read_csv(DESTINATION)
validate_predictions(saved)
pd.testing.assert_series_equal(saved['load_id'], template['load_id'])
# Saving cents can move a prediction by at most half a cent.
np.testing.assert_allclose(saved['predicted_rate'], filled['predicted_rate'], rtol=0, atol=0.005000001)

print('Saved CSV:', DESTINATION)
print('Saved final pipeline:', model_path)
display(saved['predicted_rate'].describe().to_frame('predicted_rate_USD').round(2))

Saved CSV: C:\Users\default.LAPTOP-H1OG6SJ2\Desktop\Projects\spotter\validation-predictions.csv
Saved final pipeline: C:\Users\default.LAPTOP-H1OG6SJ2\Desktop\Projects\spotter\outputs\final_catboost\pipeline.joblib


,predicted_rate_USD
count,12000.00
mean,2358.71
std,1326.77
min,449.37
25%,1286.41
50%,2044.34
75%,3351.82
max,5840.00


## 6. Verify preservation and save the run metadata

These checks confirm that the regenerated output is complete and the original inputs and evaluation results remain intact. Metadata records the exact model settings, training dates, input hashes, and output hash so the generated file can be traced back to this run.

In [8]:
assert source_hashes(ROOT) == original_hashes, 'A supplied CSV changed.'
for relative_path, digest in evaluation_hashes.items():
    assert file_hash(ROOT / relative_path) == digest, relative_path

output_hash = file_hash(DESTINATION)
reproduced_previous_output = previous_output_hash == output_hash if previous_output_hash is not None else None
checks = {
    'schema_and_ids_valid': True,
    'all_predictions_finite_positive': True,
    'model_reload_matches': True,
    'template_order_preserved': True,
    'source_csvs_unchanged': True,
    'original_evaluation_artifacts_unchanged': True,
}
metadata = {
    'model': configuration['selected_model'],
    'parameters': parameters,
    'training_rows': len(labeled),
    'training_start': labeled['date'].min().date().isoformat(),
    'training_end': labeled['date'].max().date().isoformat(),
    'prediction_rows': len(saved),
    'output': DESTINATION.name,
    'output_columns': list(saved.columns),
    'alignment': 'Matched by load_id; original template order preserved',
    'price_precision': 'USD, two decimal places',
    'source_sha256': original_hashes,
    'output_sha256': output_hash,
    'catboost_version': catboost.__version__,
    'checks': checks,
    'generated_by': 'notebooks/04_final_validation_predictions.ipynb',
    'previous_output_reproduced_exactly': reproduced_previous_output,
    'evaluation_note': 'Refitted on all January-October labels. Earlier test scores describe the January-August evaluation model; November-December true prices are unavailable.',
}
metadata_path = ARTIFACT_DIR / 'metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
display(pd.DataFrame({'check':list(checks), 'passed':list(checks.values())}))
print('Output SHA-256:', output_hash)
if previous_output_hash is not None:
    print('Regenerated CSV is byte-for-byte identical to the previous result:', reproduced_previous_output)
print('Metadata saved:', metadata_path)

,check,passed
0,schema_and_ids_valid,True
1,all_predictions_finite_positive,True
2,model_reload_matches,True
3,template_order_preserved,True
4,source_csvs_unchanged,True
5,original_evaluation_artifacts_unchanged,True


Output SHA-256: d870496610c781065b69b2db1d0479b2b55d6a67e1401b2c0c86792ad53947b8
Regenerated CSV is byte-for-byte identical to the previous result: True
Metadata saved: C:\Users\default.LAPTOP-H1OG6SJ2\Desktop\Projects\spotter\outputs\final_catboost\metadata.json


## Completed output

The file below is the requested filled template. The notebook has actually trained the final model and regenerated it during this run. No new accuracy percentage is claimed for the final inputs because their true prices are hidden.

In [9]:
display(Markdown(
    f"**Created `{DESTINATION.name}` with {len(saved):,} complete predictions.** "
    f"Columns: `{', '.join(saved.columns)}`. "
    f"The final model used {len(labeled):,} labeled rows through {labeled['date'].max().date()}."
))
print('CSV size (bytes):', DESTINATION.stat().st_size)
print(f'Prediction range: ${saved.predicted_rate.min():,.2f} to ${saved.predicted_rate.max():,.2f}')
assert source_hashes(ROOT) == original_hashes

**Created `validation-predictions.csv` with 12,000 complete predictions.** Columns: `load_id, predicted_rate`. The final model used 48,000 labeled rows through 2025-10-31.

CSV size (bytes): 226136
Prediction range: $449.37 to $5,840.00
